In [86]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact

In [87]:
high_TE_density_regions = ["NC_092080.1;24000000;26000000", "NC_092081.1;1;2000000", "NC_092082.1;26000000;44000000", "NC_092082.1;45000000;60000000", "NC_092083.1;1;2000000", "NC_092084.1;25000000;32000000", "NC_092085.1;1;12000000"]

regions = pd.DataFrame(
    [r.split(";") for r in high_TE_density_regions],
    columns=["seqid", "start", "end"]
)

regions["start"] = regions["start"].astype(int)
regions["end"] = regions["end"].astype(int)
regions["region_id"] = (
    regions["seqid"] + ";" +
    regions["start"].astype(str) + ";" +
    regions["end"].astype(str)
)
regions.head()

,seqid,start,end,region_id
0,NC_092080.1,24000000,26000000,NC_092080.1;24000000;26000000
1,NC_092081.1,1,2000000,NC_092081.1;1;2000000
2,NC_092082.1,26000000,44000000,NC_092082.1;26000000;44000000
3,NC_092082.1,45000000,60000000,NC_092082.1;45000000;60000000
4,NC_092083.1,1,2000000,NC_092083.1;1;2000000


In [88]:
suzukii_te_annotation = pd.read_csv(
    "suzukii_te_annotation.tsv", sep="\t")
suzukii_te_annotation = suzukii_te_annotation.rename(columns={"Sequence_Name": "seqid",
                                                              "Alignment_Start" : "start",
                                                              "Alignment_End" : "end"})
suzukii_te_annotation.head()

,Repeat_ID,Divergence,Deletion,Insertion,seqid,start,end,Element_Length,Orientation,Family_Name,Class,Base_Count,5'_Fragment_Length,3'_Fragment_Length,Additional_Info,Repeat_Count,Score
0,10299,1.300,3.000,0.600,NC_092080.1,21824,23072,1269,C,nf_292_LTR_1,LTR/LTR,(0),1269,1,28,1,No_ref_available
1,2536/284,7.711,2.253,1.259,NC_092080.1,35610,44782,402,C,nf_26_COPIA_1,LTR/COPIA,(352),8153,4236,43/58,2,No_ref_available
2,886/366,16.849,0.000,0.000,NC_092080.1,35924,36254,214,+,nf_25_CR1_1,LINE/CR1,1565,1779,(6831),44/44,2,0.025
3,523,19.000,18.100,1.200,NC_092080.1,36339,36482,168,+,nf_205_CR1_2,LINE/CR1,1763,1930,(37),46,1,0.085
4,578,7.200,12.100,1.800,NC_092080.1,36975,37073,109,C,Gypsy-30_DWil-I_1,LTR/GYPSY,(261),5618,5510,48,1,No_ref_available


In [89]:
suzukii_genome_annotation = pd.read_csv(
    "suzukii_transfer_annotation_v2.gff", sep="\t", comment="#", header=None)
suzukii_genome_annotation.columns = [
    "seqid", "source", "type",
    "start", "end", "score",
    "strand", "phase", "attributes"
]

suzukii_genome_annotation.head()

,seqid,source,type,start,end,score,strand,phase,attributes
0,|,Liftoff,gene,183,251,.,+,.,ID=gene-Dmel_CR34062;Dbxref=FLYBASE:FBgn001370...
1,NC_060762.1,Liftoff,tRNA,183,251,.,+,.,ID=rna-Dmel_CR34062;Parent=gene-Dmel_CR34062;D...
2,NC_060762.1,Liftoff,exon,183,251,.,+,.,ID=exon-Dmel_CR34062-1;Parent=rna-Dmel_CR34062...
3,NC_060762.1,Liftoff,gene,252,1275,.,+,.,ID=gene-Dmel_CG34063;Dbxref=FLYBASE:FBgn001368...
4,NC_060762.1,Liftoff,mRNA,252,1275,.,+,.,ID=rna-Dmel_CG34063;Parent=gene-Dmel_CG34063;D...


In [90]:
def parse_attributes(attr):
    d = {}
    for item in attr.split(";"):
        if "=" in item:
            key, value = item.split("=", 1)
            d[key] = value
    return d

suzukii_genome_annotation["attr_dict"] = suzukii_genome_annotation["attributes"].apply(parse_attributes)

suzukii_genome_annotation["gene_id"] = suzukii_genome_annotation["attr_dict"].apply(lambda x: x.get("ID"))
suzukii_genome_annotation["gene_name"] = suzukii_genome_annotation["attr_dict"].apply(lambda x: x.get("Name"))
suzukii_genome_annotation["gene_id"] = (
    suzukii_genome_annotation["gene_id"]
    .str.replace(r"^(gene|rna|exon)-", "", regex=True)
)
suzukii_genome_annotation.head()

,seqid,source,type,start,end,score,strand,phase,attributes,attr_dict,gene_id,gene_name
0,|,Liftoff,gene,183,251,.,+,.,ID=gene-Dmel_CR34062;Dbxref=FLYBASE:FBgn001370...,"{'ID': 'gene-Dmel_CR34062', 'Dbxref': 'FLYBASE...",Dmel_CR34062,trnM
1,NC_060762.1,Liftoff,tRNA,183,251,.,+,.,ID=rna-Dmel_CR34062;Parent=gene-Dmel_CR34062;D...,"{'ID': 'rna-Dmel_CR34062', 'Parent': 'gene-Dme...",Dmel_CR34062,None
2,NC_060762.1,Liftoff,exon,183,251,.,+,.,ID=exon-Dmel_CR34062-1;Parent=rna-Dmel_CR34062...,"{'ID': 'exon-Dmel_CR34062-1', 'Parent': 'rna-D...",Dmel_CR34062-1,None
3,NC_060762.1,Liftoff,gene,252,1275,.,+,.,ID=gene-Dmel_CG34063;Dbxref=FLYBASE:FBgn001368...,"{'ID': 'gene-Dmel_CG34063', 'Dbxref': 'FLYBASE...",Dmel_CG34063,ND2
4,NC_060762.1,Liftoff,mRNA,252,1275,.,+,.,ID=rna-Dmel_CG34063;Parent=gene-Dmel_CG34063;D...,"{'ID': 'rna-Dmel_CG34063', 'Parent': 'gene-Dme...",Dmel_CG34063,None


In [91]:
suzukii_te_annotation["te_id"] = suzukii_te_annotation.index

In [92]:
te_in_regions = []

for _, reg in regions.iterrows():
    subset = suzukii_te_annotation[
        (suzukii_te_annotation["seqid"] == reg["seqid"]) &
        (suzukii_te_annotation["start"] <= reg["end"]) &
        (suzukii_te_annotation["end"] >= reg["start"])
    ].copy()

    subset["te_id"] = subset.index
    subset["region_id"] = f"{reg['seqid']};{reg['start']};{reg['end']}"
    te_in_regions.append(subset)

te_in_regions_df = pd.concat(te_in_regions, ignore_index=True)
te_in_regions_df.head()

,Repeat_ID,Divergence,Deletion,Insertion,seqid,start,end,Element_Length,Orientation,Family_Name,Class,Base_Count,5'_Fragment_Length,3'_Fragment_Length,Additional_Info,Repeat_Count,Score,te_id,region_id
0,490,19.600,1.300,0.600,NC_092080.1,24020094,24020252,160,+,nf_158_LTR_1,LTR/LTR,1,160,(2668),16211,1,No_ref_available,1709,NC_092080.1;24000000;26000000
1,280/302,18.352,2.337,0.627,NC_092080.1,24041814,24042196,89,C,nf_52_LTR_1,LTR/BELPAO,(6029),125,33,16231/16233,2,No_ref_available,1710,NC_092080.1;24000000;26000000
2,629,18.400,15.200,3.400,NC_092080.1,24042178,24042335,176,C,nf_185_TIR_2,DNA/DNA,(49),2154,1979,16235,1,0.080,1711,NC_092080.1;24000000;26000000
3,704,10.900,1.800,0.000,NC_092080.1,24051247,24051356,112,+,Gypsy-6_DBi-I,LTR/GYPSY,5020,5131,(117),16238,1,No_ref_available,1712,NC_092080.1;24000000;26000000
4,1711,12.400,4.800,0.300,NC_092080.1,24051370,24051660,304,+,nf_10_TIR_1,DNA/DNA,19,322,(0),16239,1,0.944,1713,NC_092080.1;24000000;26000000


In [93]:
suzukii_te_annotation["in_region"] = False
suzukii_te_annotation["te_id"] = suzukii_te_annotation.index
suzukii_te_annotation["in_region"] = suzukii_te_annotation["te_id"].isin(
    te_in_regions_df["te_id"]
)
background_df = suzukii_te_annotation[~suzukii_te_annotation["in_region"]]

target_counts = te_in_regions_df["Class"].value_counts()
background_counts = background_df["Class"].value_counts()

counts = pd.concat([target_counts, background_counts], axis=1)
counts.columns = ["target", "background"]
counts = counts.fillna(0)

In [94]:
total_target = counts["target"].sum()
total_background = counts["background"].sum()

In [95]:
results_by_region = []

for reg_id in regions["region_id"]:

    seqid, start, end = reg_id.split(";")
    start = int(start)
    end = int(end)

    region_te = suzukii_te_annotation[
        (suzukii_te_annotation["seqid"] == seqid) &
        (suzukii_te_annotation["start"] <= end) &
        (suzukii_te_annotation["end"] >= start)
]

    chrom_te = suzukii_te_annotation[suzukii_te_annotation["seqid"] == seqid]
    background_te = chrom_te[~((chrom_te["start"] <= end) & (chrom_te["end"] >= start))]

    target_counts = region_te["Class"].value_counts()
    background_counts = background_te["Class"].value_counts()
    counts = pd.concat([target_counts, background_counts], axis=1)
    counts.columns = ["target", "background"]
    counts = counts.fillna(0)

    total_target = counts["target"].sum()
    total_background = counts["background"].sum()

    for te_class in counts.index:
        a = counts.loc[te_class, "target"]
        c = counts.loc[te_class, "background"]
        b = total_target - a
        d = total_background - c

        table = [[a + 0.5, b + 0.5], [c + 0.5, d + 0.5]]

        oddsratio, pvalue = fisher_exact(table)

        results_by_region.append({
            "region": reg_id,
            "Class": te_class,
            "odds_ratio": oddsratio,
            "p_value": pvalue
        })

results_by_region_df = pd.DataFrame(results_by_region)
results_by_region_df.to_csv("Te_density_suzukiii.csv")

In [96]:
sig = results_by_region_df[
    (results_by_region_df["p_value"] < 0.05) &
    (abs(np.log2(results_by_region_df["odds_ratio"])) > 0.5)
]
sig.to_csv("Te_density_suzukiii.csv")

G:\Programas\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log2
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [114]:
chromosomes = {
    "NC_092080.1": "2L",
    "NC_092081.1": "2R",
    "NC_092082.1": "3",
    "NC_092083.1": "4",
    "NC_092084.1": "X",
    "NC_092085.1": "Y"
}

suzukii_te_annotation = suzukii_te_annotation[
    suzukii_te_annotation["seqid"].isin(chromosomes.keys())
]

In [115]:
suzukii_te_annotation["length"] = (
    suzukii_te_annotation["end"] - suzukii_te_annotation["start"]
)
density.append(len(subset))

In [116]:
window_size = 100_000

windows = []

for chrom in chromosomes.keys():
    chrom_data = suzukii_te_annotation[
        suzukii_te_annotation["seqid"] == chrom
    ]

    max_pos = chrom_data["end"].max()

    for start in range(0, int(max_pos), window_size):
        end = start + window_size
        windows.append([chrom, start, end])

windows_df = pd.DataFrame(windows, columns=["seqid", "start", "end"])

In [122]:
density_data = []

for _, win in windows_df.iterrows():
    subset = suzukii_te_annotation[
        (suzukii_te_annotation["seqid"] == win["seqid"]) &
        (suzukii_te_annotation["start"] <= win["end"]) &
        (suzukii_te_annotation["end"] >= win["start"])
    ]

    grouped = subset.groupby("Class_simple")["length"].sum()

    row = {
        "seqid": win["seqid"],
        "start": win["start"]
    }

    for cls in ["LTR", "LINE", "DNA", "RC"]:
        row[cls] = grouped.get(cls, 0)

    density_data.append(row)

density_df = pd.DataFrame(density_data)


In [123]:
windows_df["density_smooth"] = windows_df.groupby("seqid")["density"].transform(
    lambda x: x.rolling(window=5, center=True, min_periods=1).mean()
)

In [124]:
for cls in ["LTR", "LINE", "DNA", "RC"]:
    density_df[cls + "_smooth"] = density_df.groupby("seqid")[cls].transform(
        lambda x: x.rolling(window=5, center=True, min_periods=1).mean()
    )

In [120]:
suzukii_te_annotation["Class_simple"] = (
    suzukii_te_annotation["Class"].str.split("/").str[0]
)